for whole convo

In [ ]:
# Get all messages from first conversation
messages = [m for m in items[0]["all_messages_events"] if m.get("role") != "system"]

# Extract language from first user message
language = next((m["content"].split("**Selected Language:**")[1].split("\n")[0].strip() 
                 for m in messages 
                 if m.get("role") == "user" and "**Selected Language:**" in m.get("content", "")), None)

# Build text for each user query in one pass
all_texts = []
current_conversation = None
current_user_query = None

for i, msg in enumerate(messages):
    role = msg.get("role")
    
    # When we encounter a user message, start a new conversation
    if role == "user":
        # If there was a previous conversation, save it
        if current_conversation is not None:
            all_texts.append("\n".join(current_conversation))
        
        # Extract user query
        user_content = msg["content"]
        if '"' in user_content:
            try:
                user_query = user_content.split('"')[1]
            except:
                user_query = user_content
        else:
            user_query = user_content
        
        # Start new conversation
        current_user_query = user_query
        current_conversation = [
            f"User Query: {user_query}",
            f"Language: {language}",
            "=" * 50
        ]
    
    # Process assistant tool calls
    elif role == "assistant" and "tool_calls" in msg:
        for tool_call in msg["tool_calls"]:
            tool_name = tool_call["function"]["name"]
            tool_args = tool_call["function"]["arguments"]
            current_conversation.append(f"\nAssistant called tool: {tool_name}")
            current_conversation.append(f"Arguments: {tool_args}")
    
    # Process tool returns
    elif role == "tool":
        tool_name = msg.get("name", "")
        tool_result = msg.get("content", "")
        current_conversation.append(f"\nTool '{tool_name}' returned:")
        current_conversation.append(tool_result[:500] + "..." if len(tool_result) > 500 else tool_result)
        current_conversation.append("=" * 50)
    
    # Process final assistant response (has content, no tool_calls)
    elif role == "assistant" and "content" in msg and msg.get("content") and "tool_calls" not in msg:
        current_conversation.append("=" * 50)
        current_conversation.append(f"\nFinal Response:")
        current_conversation.append(msg["content"])

# Don't forget the last conversation
if current_conversation is not None:
    all_texts.append("\n".join(current_conversation))

# Now all_texts contains one text string per user query
# You can iterate through them to evaluate each one
for idx, text in enumerate(all_texts):
    print(f"\n{'='*60}")
    print(f"Query {idx + 1} of {len(all_texts)}")
    print(f"{'='*60}")
    print(text)
    print("\n")
    
    # Pass to LLM for evaluation
    # eval_result = await evaluation_agent.run(text)